# 1. IMPORT LIBRARIES


In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 2. LOAD DATA

In [2]:
df = pd.read_csv("car_maintenance_dataset_expanded.csv")

# 3. CLEAN DATA

In [3]:
df = df[(df["km_diff"] >= 0) & (df["days_diff"] > 0)]

# 4. ENCODE PART (One-Hot)

In [4]:
df = pd.get_dummies(df, columns=["part"])

part_cols = [c for c in df.columns if c.startswith("part_")]

# 5. CREATE TARGET 

In [5]:
# urgency score = how "close to failure"

# Normalize km and days globally (NOT per part)
df["km_norm"]   = df["km_diff"]   / df["km_diff"].max()
df["days_norm"] = df["days_diff"] / df["days_diff"].max()

# Final urgency score (continuous target)
df["urgency"] = 0.7 * df["km_norm"] + 0.3 * df["days_norm"]

# 6. FEATURES & TARGET

In [6]:
FEATURES = [
    "km_diff",
    "days_diff",
    "current_km"
] + part_cols

X = df[FEATURES]
y = df["urgency"]

# 7. TRAIN / TEST SPLIT

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 8. TRAIN MODEL

In [8]:
model2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model2.fit(X_train, y_train)

RandomForestRegressor(max_depth=12, n_estimators=200, n_jobs=-1,
                      random_state=42)

# 9. EVALUATION


In [9]:
y_pred = model2.predict(X_test)

print("\n=== MODEL 2 (URGENCY) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


=== MODEL 2 (URGENCY) ===
RMSE: 0.0020784775998128027
R2 Score: 0.9993609469094067


# 10. SAVE MODEL

In [10]:
joblib.dump(model2, "parts_label_encoder.pkl")
joblib.dump(FEATURES, "model2_features.pkl")

print("\n✅ Model 2 saved!")


✅ Model 2 saved!



# 11. RECOMMENDATION FUNCTION (CORE)



In [11]:

def recommend_parts_with_urgency(car_data):
    """
    car_data: list of dicts (all parts of a car)

    Returns:
    sorted parts by urgency
    """

    df_input = pd.DataFrame(car_data)

    # one-hot encode
    df_input = pd.get_dummies(df_input, columns=["part"])

    # ensure same columns
    for col in FEATURES:
        if col not in df_input:
            df_input[col] = 0

    df_input = df_input[FEATURES]

    # predict urgency
    urgencies = model2.predict(df_input)

    results = []
    for i, part in enumerate(car_data):
        results.append({
            "part": part["part"],
            "urgency": float(urgencies[i])
        })

    # sort by urgency
    results.sort(key=lambda x: x["urgency"], reverse=True)

    return results

# 🔥 12. FINAL PIPELINE (Model 1 + Model 2)

In [13]:
model1 = joblib.load("need_maintainance.pkl")

def full_pipeline(car_id, car_data):

    df_input = pd.DataFrame(car_data)
    df_input = pd.get_dummies(df_input, columns=["part"])

    # ensure same features as model1
    for col in FEATURES:
        if col not in df_input:
            df_input[col] = 0

    df_input = df_input[FEATURES]

    # Step 1 → Model 1
    preds = model1.predict(df_input)

    if preds.sum() == 0:
        return {
            "car_id": car_id,
            "needs_maintenance": False,
            "recommended_parts": []
        }

    # Step 2 → Model 2 ranking
    ranked = recommend_parts_with_urgency(car_data)

    # take only top urgent parts
    TOP_K = 3
    THRESHOLD = 0.05

    recommended = [p["part"] for p in ranked if p["urgency"] > THRESHOLD][:TOP_K]

    return {
        "car_id": car_id,
        "needs_maintenance": True,
        "recommended_parts": recommended,
        "ranked_parts": ranked
    }


# 13. TEST

In [14]:
test_car = [
    {"part": "brake_pads", "km_diff": 25000, "days_diff": 200, "current_km": 90000},
    {"part": "air_filter", "km_diff": 12000, "days_diff": 100, "current_km": 90000},
    {"part": "battery", "km_diff": 2000, "days_diff": 800, "current_km": 90000},
]

result = full_pipeline(5162, test_car)

print("\n=== FINAL RESULT ===")
print(result)


=== FINAL RESULT ===
{'car_id': 5162, 'needs_maintenance': True, 'recommended_parts': ['brake_pads'], 'ranked_parts': [{'part': 'brake_pads', 'urgency': 0.08996086179076805}, {'part': 'air_filter', 'urgency': 0.04302604651634102}, {'part': 'battery', 'urgency': 0.027475465143423885}]}
